In [1]:
import torch
import torch.nn as nn

class Conv1DBranch(nn.Module):
    def __init__(self, input_len, D):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.BatchNorm1d(input_len),
            nn.Conv1d(input_len, D, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(D),
            nn.MaxPool1d(2),
            nn.Conv1d(D, 2 * D, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(2 * D),
            nn.MaxPool1d(2),
            nn.Conv1d(2 * D, 4 * D, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(4 * D),
        )
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(4 * D, input_len)

    def forward(self, x):
        x = self.cnn(x)
        x = self.global_avg_pool(x).squeeze(-1)  # [B, C]
        x = self.fc(x)
        return x

# Example
B, input_len, L, D = 55, 2, 15, 16   # batch, channels, sequence length, base filters
model = Conv1DBranch(input_len, D)

x = torch.randn(B, input_len, L)  # [4, 8, 32]
out = model(x)

print("Input shape:", x.shape)
print("Output shape:", out.shape)
print(model)


Input shape: torch.Size([55, 2, 15])
Output shape: torch.Size([55, 2])
Conv1DBranch(
  (cnn): Sequential(
    (0): BatchNorm1d(2, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (1): Conv1d(2, 16, kernel_size=(3,), stride=(1,), padding=(1,))
    (2): ReLU()
    (3): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (4): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv1d(16, 32, kernel_size=(3,), stride=(1,), padding=(1,))
    (6): ReLU()
    (7): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (10): ReLU()
    (11): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (global_avg_pool): AdaptiveAvgPool1d(output_size=1)
  (fc): Linear(in_features=64, out_features=2, bias=True)

In [2]:
import torch
import torch.nn as nn

class Conv1DDecoder(nn.Module):
    def __init__(self, input_len, D, output_len):
        super().__init__()
        self.fc = nn.Linear(input_len, 4 * D)   # match encoder bottleneck
        
        self.upcnn = nn.Sequential(
            nn.ConvTranspose1d(4 * D, 2 * D, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(2 * D),

            nn.ConvTranspose1d(2 * D, D, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(D),
        )
        self.global_avg_pool = nn.AdaptiveAvgPool1d(output_len)

        self.cnn_shaping = nn.Conv1d(D, 1, kernel_size=3, padding=1)

    def forward(self, x):
        # x: [B, input_len] (latent from encoder)
        x = self.fc(x)                      # [B, 4D]
        x = x.unsqueeze(-1)                 # [B, 4D, 1]
        x = self.upcnn(x)                   # [B, output_len, L]
        x = self.global_avg_pool(x)
        x = self.cnn_shaping(x)
        return x


In [3]:
B = 55
D = 32

input_len = 3
input_len_2 = 15

# B, input_len, L, D = 55, 2, 15, 16 
# B, input_len, L, D = 55, 2, 15, 16 

output_len = 13

encoder = Conv1DBranch(input_len, D)
decoder = Conv1DDecoder(input_len, D, output_len)

x = torch.randn(B, input_len, L)
z = encoder(x)              # [B, 8]
x_hat = decoder(z)           # [B, 8, 32]

print("Input shape :", x.shape)
print("Latent shape:", z.shape)
print("Decoded shape:", x_hat.shape)


Input shape : torch.Size([55, 3, 15])
Latent shape: torch.Size([55, 3])
Decoded shape: torch.Size([55, 1, 13])


In [69]:
# ======================================================================================================================
# Example usage
batch_size = 8

D = 16
# T = 100

# input_channels_1 = 5  # Number of input channels

input_channels_2 = 1  
height_length_2 = 54

# input_channels_3 = 1  # Number of input channels
# height_length_3 = 27
# width_length_3 = 33

output_shape = [[7]]

x_init = [ 
    # torch.randn(input_channels_1, T), # time series evolving in time
      torch.randn(input_channels_2, height_length_2), # profiles evolving in time
    #   torch.randn(input_channels_3, T, height_length_3, width_length_3) # images evolving in time
      ]

model = Conv1DBranch( input_channels_2 , D)

x = [ 
    # torch.randn(batch_size, input_channels_1, T), # time series evolving in time
      torch.randn(batch_size, input_channels_2, height_length_2), # profiles evolving in time
    #   torch.randn(batch_size, input_channels_3, T, height_length_3, width_length_3) # images evolving in time
]
print(x[0].shape)
output = model(x[0])


torch.Size([8, 1, 54])


In [70]:
print(output.shape)

torch.Size([8, 1])


In [ ]:
import torch
import torch.nn as nn



# ======================================================================================================================
class Numerical0DBranch(nn.Module):

    # ------------------------------------------------------------------------------------------------------------------
    def __init__(self, input_shape):
        super().__init__()

        self.n_profiles = input_shape[0]

        self.bn = nn.BatchNorm1d(self.n_profiles)

    # ------------------------------------------------------------------------------------------------------------------
    def forward(self, x):
        return self.bn(x)

# ======================================================================================================================
class Conv1DBranch(nn.Module):

    # ------------------------------------------------------------------------------------------------------------------
    def __init__(self, input_shape, D):
        super().__init__()

        self.n_profiles = input_shape[0]

        self.cnn = nn.Sequential(
            nn.BatchNorm1d(self.n_profiles),
            nn.Conv1d(self.n_profiles, D, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(D),
            nn.MaxPool1d(2),
            nn.Conv1d(D, 2 * D, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(2 * D),
            nn.MaxPool1d(2),
            nn.Conv1d(2 * D, 4 * D, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(4 * D),
        )
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(4 * D, self.n_profiles)

    # ------------------------------------------------------------------------------------------------------------------
    def forward(self, x):
        print(x.shape)
        x = self.cnn(x)
        print(x.shape)
        x = self.global_avg_pool(x).squeeze(-1)  # [B, C]
        print(x.shape)
        x = self.fc(x)
        print(x.shape)
        return x

# ======================================================================================================================
class Conv2DBranch(nn.Module):

    # ------------------------------------------------------------------------------------------------------------------
    def __init__(self, input_shape, D):
        super().__init__()

        self.n_profiles = input_shape[0]
        
        self.cnn = nn.Sequential(
            nn.BatchNorm2d(self.n_profiles),
            nn.Conv2d(self.n_profiles, D, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(D),
            nn.MaxPool2d(2),
            nn.Conv2d(D, 2 * D, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(2 * D),
            nn.MaxPool2d(2),
            nn.Conv2d(2 * D, 4 * D, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(4 * D),
        )
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(4 * D, self.n_profiles)

    # ------------------------------------------------------------------------------------------------------------------
    def forward(self, x):
        x = self.cnn(x)
        x = self.global_avg_pool(x).squeeze(-1).squeeze(-1)  # [B, C]
        x = self.fc(x)
        return x


# ======================================================================================================================
class DecoderNumerical0DBranch(nn.Module):

    # ------------------------------------------------------------------------------------------------------------------
    def __init__(self, merged_input_dim, output_shape, D):
        super().__init__()

        self.n_timeseries = output_shape[0]

        self.fc = nn.Sequential(
            nn.BatchNorm1d(merged_input_dim),
            nn.Linear(merged_input_dim, 4 * D),
            nn.ReLU(),
            nn.BatchNorm1d(4 * D),
            nn.Linear(4 * D, 2 * D),
            nn.ReLU(),
            nn.BatchNorm1d(2 * D),
            nn.Dropout(0.2),
            nn.Linear(2 * D, output_len),
        )

    # ------------------------------------------------------------------------------------------------------------------
    def forward(self, x):
        x = self.fc(x)
        return x

# ======================================================================================================================
class Conv1DDecoder(nn.Module):
    def __init__(self, merged_input_dim, output_shape, D):
        super().__init__()

        self.n_profiles = output_shape[0]
        self.height_profiles = output_shape[1]

        self.fc = nn.Linear(merged_input_dim, 4 * D * self.height_profiles )   # match encoder bottleneck
        
        self.transposecnn = nn.Sequential(
            nn.ConvTranspose1d(4 * D, 2 * D, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(2 * D),

            nn.ConvTranspose1d(2 * D, D, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(D),

            nn.ConvTranspose1d(D, self.n_profiles, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(self.n_profiles),
        )

        self.global_avg_pool = nn.AdaptiveAvgPool1d(self.height_profiles)

        # self.cnn_shaping = nn.Conv1d(D, self.n_profiles, kernel_size=3, padding=1)

    def forward(self, x):
        print(x.shape)
        x = self.fc(x)                      # [B, 4D]
        print(x.shape)
        x = x.view(B, 4*D, self.height_profiles)     
        print(x.shape)
        x = self.transposecnn(x)                   # [B, output_len, L]
        print(x.shape)
        x = self.global_avg_pool(x)
        return x
    

# ======================================================================================================================
class Conv2DDecoder(nn.Module):
    def __init__(self, merged_input_dim, output_shape, D):
        super().__init__()

        self.n_profiles = output_shape[0]
        self.height_profiles = output_shape[1]
        self.weight_profiles = output_shape[2]

        self.fc = nn.Linear(merged_input_dim, 4 * D * self.height_profiles * self.weight_profiles )   # match encoder bottleneck
        
        self.transposecnn1 = nn.Sequential(
            nn.ConvTranspose2d(4 * D, 2 * D, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(2 * D),
        )

        self.transposecnn2 = nn.Sequential(
            nn.ConvTranspose2d(2 * D, D, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(D),
        )

        self.transposecnn3 = nn.Sequential(
            nn.ConvTranspose2d(D, self.n_profiles, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(self.n_profiles),
        )

        self.global_avg_pool = nn.AdaptiveAvgPool2d((self.height_profiles, self.weight_profiles))

        # self.cnn_shaping = nn.Conv1d(D, self.n_profiles, kernel_size=3, padding=1)

    def forward(self, x):
        print(x.shape)
        x = self.fc(x)                      # [B, 4D]
        print(x.shape)
        x = x.view(B, 4*D, self.height_profiles, self.weight_profiles)     
        print("after view reshape", x.shape)
        x = self.transposecnn1(x)                   # [B, output_len, L]
        print(x.shape)
        x = self.transposecnn2(x)                   # [B, output_len, L]
        print(x.shape)
        x = self.transposecnn3(x)                   # [B, output_len, L]
        print(x.shape)
        x = self.global_avg_pool(x)
        return x


class MultiBranchCNNModel(nn.Module):

    # ------------------------------------------------------------------------------------------------------------------
    def __init__(self, input_shapes, output_shapes, D=16):
        super().__init__()

        self.input_branches = nn.ModuleList()
        merged_input_dim = 0

        for shape in input_shapes:
            if len(shape) == 3:  # e.g., (2, 15, 17)
                branch = Conv2DBranch(shape, D)
                merged_input_dim += shape[0]
            elif len(shape) == 2:  # e.g., (1, 15)
                branch = Conv1DBranch(shape, D)
                merged_input_dim += shape[0]
            elif len(shape) == 1:  # e.g., (7,)
                branch = Numerical0DBranch(shape)
                merged_input_dim += shape[0]
            else:
                raise ValueError(f"Unsupported input shape: {shape}")
            self.input_branches.append(branch)
        
        # self.output_shape = output_shape[0][0]
        self.output_branches = nn.ModuleList()

        for shape in output_shapes:
            if len(shape) == 3:  # e.g., (2, 15, 17)
                print("2D decoding branch for ", shape)
                branch = Conv2DDecoder(merged_input_dim, shape, D)
            elif len(shape) == 2:  # e.g., (1, 15)
                print("1D decoding branch for ", shape)
                branch = Conv1DDecoder(merged_input_dim, shape, D)
            elif len(shape) == 1:  # e.g., (7,)
                print("0D decoding branch for ", shape)
                branch = DecoderNumerical0DBranch(merged_input_dim, shape, D)
            else:
                raise ValueError(f"Unsupported input shape: {shape}")
            self.output_branches.append(branch)
        
        print("len (self.output_branches) is ", len (self.output_branches))

    # ------------------------------------------------------------------------------------------------------------------
    def forward(self, *inputs):
        
        encoded_representation = []

        for branch, x in zip(self.input_branches, inputs):
            out = branch(x)
            encoded_representation.append(out)

        merged = torch.cat(encoded_representation, dim=1)

        decoded_representation = []

        for branch in self.output_branches :
            out = branch(merged)
            decoded_representation.append(out)
        
        return decoded_representation

    # ------------------------------------------------------------------------------------------------------------------


In [ ]:
B=32

input_shapes = [(6,), (2, 12), (3, 8), (3, 65, 65)]
output_shapes = [(7,), (1, 23), (1, 23, 45), (22, 76, 45)]
D = 16

model = MultiBranchCNNModel(input_shapes, output_shapes, D)
model

import torch

# Prepend batch dimension dynamically
input = ([torch.randn((B,) + shape) for shape in input_shapes])

print( "\nINPUT SHAPES: ", [ arr.shape for arr in input ] )
print( "\nOUTPUT SHAPES: ", [ arr.shape for arr in model(*input) ] )

from torchinfo import summary
summary(model, input_size=((32, 6,), (32, 2, 12), (32, 3, 8), (32, 3, 65, 65)))


1D decoding branch for  (1, 23)
2D decoding branch for  (1, 23, 45)
2D decoding branch for  (22, 76, 45)
len (self.output_branches) is  3

INPUT SHAPES:  [torch.Size([32, 6]), torch.Size([32, 2, 12]), torch.Size([32, 3, 8]), torch.Size([32, 3, 65, 65])]
torch.Size([32, 2, 12])
torch.Size([32, 64, 3])
torch.Size([32, 64])
torch.Size([32, 2])
torch.Size([32, 3, 8])
torch.Size([32, 64, 2])
torch.Size([32, 64])
torch.Size([32, 3])
torch.Size([32, 14])
torch.Size([32, 1472])
torch.Size([32, 64, 23])
torch.Size([32, 1, 177])
torch.Size([32, 14])
torch.Size([32, 66240])
after view reshape torch.Size([32, 64, 23, 45])
torch.Size([32, 32, 45, 89])
torch.Size([32, 16, 89, 177])
torch.Size([32, 1, 177, 353])
torch.Size([32, 14])
torch.Size([32, 218880])
after view reshape torch.Size([32, 64, 76, 45])
torch.Size([32, 32, 151, 89])
torch.Size([32, 16, 301, 177])
torch.Size([32, 22, 601, 353])

OUTPUT SHAPES:  [torch.Size([32, 1, 23]), torch.Size([32, 1, 23, 45]), torch.Size([32, 22, 76, 45])]
torch

Layer (type:depth-idx)                        Output Shape              Param #
MultiBranchCNNModel                           [32, 1, 23]               --
├─ModuleList: 1-1                             --                        --
│    └─Numerical0DBranch: 2-1                 [32, 6]                   --
│    │    └─BatchNorm1d: 3-1                  [32, 6]                   12
│    └─Conv1DBranch: 2-2                      [32, 2]                   --
│    │    └─Sequential: 3-2                   [32, 64, 3]               8,116
│    │    └─AdaptiveAvgPool1d: 3-3            [32, 64, 1]               --
│    │    └─Linear: 3-4                       [32, 2]                   130
│    └─Conv1DBranch: 2-3                      [32, 3]                   --
│    │    └─Sequential: 3-5                   [32, 64, 2]               8,166
│    │    └─AdaptiveAvgPool1d: 3-6            [32, 64, 1]               --
│    │    └─Linear: 3-7                       [32, 3]                   195
│    └─Conv2